PRÁCTICA 2: Aseguramiento de Calidad de Datos en el Sistema Hospital Salud Vital

In [388]:
from dotenv import load_dotenv
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv()

#conexion a bd
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
)

In [389]:
medicos=pd.read_sql("SELECT * FROM medicos", engine)
medicos

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
0,1,Sr(a). Alicia Bonilla,1,sr(a)..bonilla@hospitalvital.mx,Estados Unidoss
1,2,Tomás Velázquez Jaimes,5,tomás.jaimes@hospitalvital.mx,Argentina
2,3,Reina Lorena Angulo,3,reina.angulo@hospitalvital.mx,México
3,4,Soledad Trejo,1,soledad.trejo@hospitalvital.mx,España
4,5,Tomás Hermelinda Rosas Guevara,5,tomás.guevara@hospitalvital.mx,
5,6,Héctor Rendón,6,héctor.rendón@hospitalvital.mx,México
6,7,Ana María Cotto Quiñónez,1,ana.quiñónez@hospitalvital.mx,España
7,8,Maximiliano Rivera,2,maximiliano.rivera@hospitalvital.mx,Alemania
8,9,Jacinto Samuel Colón,2,jacinto.colón@hospitalvital.mx,México
9,10,Indira Jaime Ozuna Cedillo,5,indira.cedillo@hospitalvital.mx,Estados Unidos


LIMPIEZA DE MEDICOS

DETECCION ERRORES

In [390]:
print(medicos.shape)
print(medicos.isnull().sum())
print(medicos.dtypes)

(100, 5)
id_medico             0
nombre_completo       0
id_especialidad       0
correo_electronico    0
nacionalidad          0
dtype: int64
id_medico             int64
nombre_completo         str
id_especialidad       int64
correo_electronico      str
nacionalidad            str
dtype: object


In [391]:
# Nacionalidades unicas — aquí veo los typos
medicos['nacionalidad'].value_counts(dropna=False) #me doy cuenta que en mexico hay México y Mexcio. Colombia y colmbia, null, Estados Unidos, Estados Unidoss

nacionalidad
México             36
Estados Unidos     16
Argentina          14
Alemania           11
España              8
Colombia            4
Colmbia             3
Mexcio              3
Estados Unidoss     2
null                2
                    1
Name: count, dtype: int64

In [392]:
# Valores 'null' como texto y nacionalidades vacias
medicos[medicos['nacionalidad'].isin(['null', '']) | medicos['nacionalidad'].isnull()]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
4,5,Tomás Hermelinda Rosas Guevara,5,tomás.guevara@hospitalvital.mx,
44,45,Renato Carmona Espino,6,renato.espino@hospitalvital.mx,null
94,95,Óliver Nájera,3,óliver.nájera@hospitalvital.mx,null


In [393]:
mask = medicos['correo_electronico'].str.contains(r'\(|\)|mtro\.|ing\.|dr\.|lic\.|sra\.', case=False, na=False)
medicos[mask]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad
0,1,Sr(a). Alicia Bonilla,1,sr(a)..bonilla@hospitalvital.mx,Estados Unidoss
14,15,Lic. Felix Crespo,3,lic..crespo@hospitalvital.mx,Colmbia
20,21,Mtro. Amanda Arteaga,1,mtro..arteaga@hospitalvital.mx,México
22,23,Sr(a). Bianca Prado,2,sr(a)..prado@hospitalvital.mx,México
25,26,Dr. Jos Uribe,3,dr..uribe@hospitalvital.mx,España
27,28,Mtro. Sofía Casanova,5,mtro..casanova@hospitalvital.mx,Estados Unidos
30,31,Ing. Karla Sanches,3,ing..sanches@hospitalvital.mx,España
33,34,Ing. Clara Menchaca,2,ing..menchaca@hospitalvital.mx,Alemania
34,35,Mtro. Dalia Orozco,6,mtro..orozco@hospitalvital.mx,Mexcio
35,36,Mtro. Rosalia Miramontes,5,mtro..miramontes@hospitalvital.mx,México


ERRRO 1 CORREGIDO NACIONALIDADES INCOSISTENTES

In [394]:
medicos['nacionalidad'] = medicos['nacionalidad'].replace({
    'Mexcio': 'México',
    'Colmbia': 'Colombia',
    'Estados Unidoss': 'Estados Unidos'
})

# verificar que ya no existan
medicos['nacionalidad'].value_counts(dropna=False)

nacionalidad
México            39
Estados Unidos    18
Argentina         14
Alemania          11
España             8
Colombia           7
null               2
                   1
Name: count, dtype: int64

error 2 nulos o cadenas vacias en nacionalidad llenadas por nacionalidad mas comun

In [395]:
# Error 2 — 'null' como texto y cadena vacía → mexico
medicos['nacionalidad'] = medicos['nacionalidad'].replace({'null': 'México', '': 'México'})
#misma consulta que antes pero ya no salieron resultados de nacionalidades con null o cadena vacia
medicos[medicos['nacionalidad'].isin(['null', '']) | medicos['nacionalidad'].isnull()]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad


Error 3: Persona con nombre incompleto (Jos)

In [396]:
medicos["nombre_completo"].unique() #veo de primera instancia que hay un nombre incompleto (Jos Uribe)-> José Uribe

<StringArray>
[             'Sr(a). Alicia Bonilla',             'Tomás Velázquez Jaimes',
                'Reina Lorena Angulo',                      'Soledad Trejo',
     'Tomás Hermelinda Rosas Guevara',                      'Héctor Rendón',
           'Ana María Cotto Quiñónez',                 'Maximiliano Rivera',
               'Jacinto Samuel Colón',         'Indira Jaime Ozuna Cedillo',
      'Noelia Ramiro Caraballo Matos',            'Martha Carlos Jaramillo',
                  'Fernando Cisneros',                 'Amador López Olivo',
                  'Lic. Felix Crespo',   'Reynaldo Humberto Espinal Crespo',
                      'Noelia Robles',                     'Homero Salinas',
             'Aurora Orozco Santiago',                     'Fabiola Galván',
               'Mtro. Amanda Arteaga',   'Miguel Ángel Rubén Rodrígez Lira',
                'Sr(a). Bianca Prado',             'Gabino Josefina Padrón',
         'Amalia Karla Ávalos Vargas',                      'D

In [397]:
medicos.loc[medicos['id_medico'] == 26, 'nombre_completo'] = 'José Uribe'

ERROR 4: Prefijos en nombres de medicos

In [398]:
#  quitar prefijos
prefijos = r'^(?:Sr\(a\)\.|Mtro\.|Ing\.|Dr\.|Lic\.)\s*'

medicos['nombre_completo'] = medicos['nombre_completo'].str.replace(
    prefijos, '', regex=True
).str.strip()

# verificar
mask_prefijo = medicos['nombre_completo'].str.contains(prefijos, na=False)
medicos[mask_prefijo]

,id_medico,nombre_completo,id_especialidad,correo_electronico,nacionalidad


In [399]:
medicos["nombre_completo"].unique()

<StringArray>
[                    'Alicia Bonilla',             'Tomás Velázquez Jaimes',
                'Reina Lorena Angulo',                      'Soledad Trejo',
     'Tomás Hermelinda Rosas Guevara',                      'Héctor Rendón',
           'Ana María Cotto Quiñónez',                 'Maximiliano Rivera',
               'Jacinto Samuel Colón',         'Indira Jaime Ozuna Cedillo',
      'Noelia Ramiro Caraballo Matos',            'Martha Carlos Jaramillo',
                  'Fernando Cisneros',                 'Amador López Olivo',
                       'Felix Crespo',   'Reynaldo Humberto Espinal Crespo',
                      'Noelia Robles',                     'Homero Salinas',
             'Aurora Orozco Santiago',                     'Fabiola Galván',
                     'Amanda Arteaga',   'Miguel Ángel Rubén Rodrígez Lira',
                       'Bianca Prado',             'Gabino Josefina Padrón',
         'Amalia Karla Ávalos Vargas',                        

In [400]:
medicos["nombre_completo"].duplicated().any() # confirme que no hay nombres duplicados

np.False_

Error 5: Correos electrónicos mal construidos

In [401]:
import unicodedata

def quitar_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def reconstruir_correo(nombre):
    partes = nombre.split()
    primer_nombre = quitar_acentos(partes[0].lower())

    if len(partes) == 2:
        primer_apellido = quitar_acentos(partes[1].lower())
    elif len(partes) == 3:
        primer_apellido = quitar_acentos(partes[1].lower())
    elif len(partes) >= 4:
        primer_apellido = quitar_acentos(partes[2].lower())

    return f"{primer_nombre}{primer_apellido}@hospitalvital.mx"

medicos['correo_electronico'] = medicos['nombre_completo'].apply(reconstruir_correo)

# verificar que no queden acentos
medicos[['nombre_completo', 'correo_electronico']].head(20)

,nombre_completo,correo_electronico
0,Alicia Bonilla,aliciabonilla@hospitalvital.mx
1,Tomás Velázquez Jaimes,tomasvelazquez@hospitalvital.mx
2,Reina Lorena Angulo,reinalorena@hospitalvital.mx
3,Soledad Trejo,soledadtrejo@hospitalvital.mx
4,Tomás Hermelinda Rosas Guevara,tomasrosas@hospitalvital.mx
5,Héctor Rendón,hectorrendon@hospitalvital.mx
6,Ana María Cotto Quiñónez,anacotto@hospitalvital.mx
7,Maximiliano Rivera,maximilianorivera@hospitalvital.mx
8,Jacinto Samuel Colón,jacintosamuel@hospitalvital.mx
9,Indira Jaime Ozuna Cedillo,indiraozuna@hospitalvital.mx


PACIENTES

In [402]:
pacientes=pd.read_sql("SELECT * FROM pacientes", engine)
pacientes

,id_paciente,nombre_completo,fecha_nacimiento,genero,correo_electronico,nacionalidad
0,1,Wilfrido Pacheco Urbina,None,F,wilfrido.pacheco@hospitalvital.mx,México
1,2,Alejandro Rentería Bueno,None,M,alejandro.rentería@hospitalvital.mx,México
2,3,Ignacio Hernádez Mejía,1945-01-07,M,ignacio.hernádez@hotmail.com,México
3,4,Karina Serrano Abreu,2018-10-01,M,karina.serrano@hospitalvital.mx,México
4,5,Patricia Ríos,1985-06-26,F,patricia.ríos@hospitalvital.mx,Estados Unidos
5,6,Ana Escobedo Sisneros,1978-06-25,M,ana.escobedo@hospitalvital.mx,México
6,7,Silvia Samaniego Quintanilla,1936-04-07,M,silvia.samaniego@hospitalvital.mx,México
7,8,Emilia Jaramillo Ulloa,1947-01-27,F,emilia.jaramillo@hospitalvital.mx,México
8,9,Graciela Quezada Rubio,1945-10-08,F,graciela.quezada@hospitalvital.mx,México
9,10,Alicia Grijalva Covarrubias,1971-07-15,F,alicia.grijalva@hospitalvital.mx,México


In [403]:
pacientes["nacionalidad"].unique()

<StringArray>
[        'México', 'Estados Unidos',      'Argentina',       'Colombia',
         'España',          'Japón',          'Chile',       'Alemania',
        'Francia',         'Italia']
Length: 10, dtype: str

In [404]:
medicos["nacionalidad"].unique()

<StringArray>
['Estados Unidos', 'Argentina', 'México', 'España', 'Alemania', 'Colombia']
Length: 6, dtype: str

In [405]:
pacientes["nombre_completo"].duplicated().any() # no hay duplicados

np.False_

In [406]:
pacientes["fecha_nacimiento"].isnull().sum()

np.int64(25)

In [407]:
# para ver todos sin límite
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pacientes[['id_paciente', 'nombre_completo', 'genero']]

,id_paciente,nombre_completo,genero
0,1,Wilfrido Pacheco Urbina,F
1,2,Alejandro Rentería Bueno,M
2,3,Ignacio Hernádez Mejía,M
3,4,Karina Serrano Abreu,M
4,5,Patricia Ríos,F
5,6,Ana Escobedo Sisneros,M
6,7,Silvia Samaniego Quintanilla,M
7,8,Emilia Jaramillo Ulloa,F
8,9,Graciela Quezada Rubio,F
9,10,Alicia Grijalva Covarrubias,F


In [408]:
pacientes[pacientes['fecha_nacimiento'].isnull()][['id_paciente', 'nombre_completo', 'fecha_nacimiento']]

,id_paciente,nombre_completo,fecha_nacimiento
0,1,Wilfrido Pacheco Urbina,None
1,2,Alejandro Rentería Bueno,None
16,17,Elias Zamudio Sauceda,None
17,18,Celia Barrios Leal,None
20,21,Ricardo Saiz,None
36,37,Ángela Gollum,None
37,38,Susana Loera Gaytán,None
43,44,Lilia Valles Dueñas,None
60,61,Verónica Arenas Mares,None
74,75,Silvano Valentín,None


 Error 6: Nacionalidad como texto repetido en médicos y pacientes

In [409]:
# Obtener nacionalidades únicas de ambas tablas (ya limpias)
nac_medicos = medicos['nacionalidad'].dropna().unique()
nac_pacientes = pacientes['nacionalidad'].dropna().unique()

todas = pd.Series(list(set(nac_medicos) | set(nac_pacientes))).sort_values().reset_index(drop=True)

nacionalidades = pd.DataFrame({
    'id_nacionalidad': range(1, len(todas) + 1),
    'nombre_nacionalidad': todas
})

print(nacionalidades)

   id_nacionalidad nombre_nacionalidad
0                1            Alemania
1                2           Argentina
2                3               Chile
3                4            Colombia
4                5              España
5                6      Estados Unidos
6                7             Francia
7                8              Italia
8                9               Japón
9               10              México


In [410]:
# Reemplazar en medicos
medicos = medicos.merge(nacionalidades, left_on='nacionalidad', right_on='nombre_nacionalidad', how='left')
medicos = medicos.drop(columns=['nacionalidad', 'nombre_nacionalidad'])

# Reemplazar en pacientes
pacientes = pacientes.merge(nacionalidades, left_on='nacionalidad', right_on='nombre_nacionalidad', how='left')
pacientes = pacientes.drop(columns=['nacionalidad', 'nombre_nacionalidad'])

# verificar
print(medicos.columns.tolist())
print(pacientes.columns.tolist())
print(medicos[['nombre_completo', 'id_nacionalidad']].head(10))
print(pacientes[['nombre_completo', 'id_nacionalidad']].head(10))

['id_medico', 'nombre_completo', 'id_especialidad', 'correo_electronico', 'id_nacionalidad']
['id_paciente', 'nombre_completo', 'fecha_nacimiento', 'genero', 'correo_electronico', 'id_nacionalidad']
                  nombre_completo  id_nacionalidad
0                  Alicia Bonilla                6
1          Tomás Velázquez Jaimes                2
2             Reina Lorena Angulo               10
3                   Soledad Trejo                5
4  Tomás Hermelinda Rosas Guevara               10
5                   Héctor Rendón               10
6        Ana María Cotto Quiñónez                5
7              Maximiliano Rivera                1
8            Jacinto Samuel Colón               10
9      Indira Jaime Ozuna Cedillo                6
                nombre_completo  id_nacionalidad
0       Wilfrido Pacheco Urbina               10
1      Alejandro Rentería Bueno               10
2        Ignacio Hernádez Mejía               10
3          Karina Serrano Abreu             

In [411]:
medicos.head()

,id_medico,nombre_completo,id_especialidad,correo_electronico,id_nacionalidad
0,1,Alicia Bonilla,1,aliciabonilla@hospitalvital.mx,6
1,2,Tomás Velázquez Jaimes,5,tomasvelazquez@hospitalvital.mx,2
2,3,Reina Lorena Angulo,3,reinalorena@hospitalvital.mx,10
3,4,Soledad Trejo,1,soledadtrejo@hospitalvital.mx,5
4,5,Tomás Hermelinda Rosas Guevara,5,tomasrosas@hospitalvital.mx,10


In [412]:
pacientes.head() #si

,id_paciente,nombre_completo,fecha_nacimiento,genero,correo_electronico,id_nacionalidad
0,1,Wilfrido Pacheco Urbina,None,F,wilfrido.pacheco@hospitalvital.mx,10
1,2,Alejandro Rentería Bueno,None,M,alejandro.rentería@hospitalvital.mx,10
2,3,Ignacio Hernádez Mejía,1945-01-07,M,ignacio.hernádez@hotmail.com,10
3,4,Karina Serrano Abreu,2018-10-01,M,karina.serrano@hospitalvital.mx,10
4,5,Patricia Ríos,1985-06-26,F,patricia.ríos@hospitalvital.mx,6


ERROR 7 GENERO INCORRECTO SEGUN NOMBRE DE PACIENTE

In [413]:
correcciones_genero = {
    4: 'F',   # Karina
    6: 'F',   # Ana
    7: 'F',   # Silvia
    12: 'M',  # Aurelio
    15: 'M',  # Maximiliano
    17: 'M',  # Elias
    20: 'M',  # Abelardo
    23: 'M',  # Maximiliano
    25: 'F',  # Inés
    26: 'M',  # Juan Carlos
    30: 'M',  # Benito
    32: 'F',  # Vanesa
    33: 'M',  # Gabino
    42: 'F',  # Anabel
    44: 'F',  # Lilia
    48: 'F',  # Florencia
    53: 'F',  # Hermelinda
    54: 'M',  # Bernabé
    57: 'M',  # Raúl
    63: 'M',  # César
    68: 'F',  # Camila
    69: 'M',  # Espartaco
    72: 'M',  # Zacarías
    73: 'M',  # Teodoro
    74: 'M',  # Zeferino
    75: 'M',  # Silvano
    76: 'F',  # Itzel
    78: 'F',  # Clara
    80: 'M',  # Pablo
    82: 'M',  # Rodrigo
    83: 'F',  # Berta
    86: 'M',  # José Carlos
    87: 'M',  # Jorge
    93: 'M',  # Rodrigo
    94: 'F',  # Concepción
    95: 'M',  # Gerónimo
    96: 'F',  # Paola
    97: 'F',  # Blanca
    104: 'F', # Victoria
    105: 'F', # Trinidad
    108: 'F', # Sandra
    109: 'M', # Gerardo
    110: 'F', # Juana
    112: 'F', # Berta
    113: 'M', # Uriel
    114: 'F', # María Cristina
    115: 'F', # Ivonne
    116: 'F', # Barbara
    117: 'M', # Humberto
    120: 'F', # Beatriz
    121: 'F', # Ana María
    124: 'F', # Ruby
    125: 'M', # Reynaldo
    128: 'F', # Camila
    129: 'M', # Luis Miguel
    132: 'F', # Georgina
    134: 'M', # Claudio
    135: 'F', # Ángela
    139: 'F', # Irma
    144: 'F', # Esmeralda
    145: 'M', # Jaime
    146: 'F', # Rosalia
    148: 'M', # Leonardo
    150: 'F', # Citlali
    154: 'F', # Yolanda
    158: 'F', # Camila
    159: 'M', # David
    160: 'M', # Marco Antonio
    161: 'M', # Rolando
    163: 'F', # Ivonne
    164: 'F', # Araceli
    166: 'F', # Verónica
    169: 'F', # Vanesa
    173: 'M', # Mariano
    175: 'F', # Anabel
    179: 'M', # Juan Carlos
    180: 'F', # Alma
    182: 'F', # Mónica
    183: 'F', # María del Carmen
    185: 'F', # Marcela
    188: 'F', # Ivonne
    196: 'F', # Olivia
    198: 'F', # Aida
    200: 'F', # Elisa
}

for id_pac, genero in correcciones_genero.items():
    pacientes.loc[pacientes['id_paciente'] == id_pac, 'genero'] = genero

# verificar
pacientes[['id_paciente', 'nombre_completo', 'genero']]

,id_paciente,nombre_completo,genero
0,1,Wilfrido Pacheco Urbina,F
1,2,Alejandro Rentería Bueno,M
2,3,Ignacio Hernádez Mejía,M
3,4,Karina Serrano Abreu,F
4,5,Patricia Ríos,F
5,6,Ana Escobedo Sisneros,F
6,7,Silvia Samaniego Quintanilla,F
7,8,Emilia Jaramillo Ulloa,F
8,9,Graciela Quezada Rubio,F
9,10,Alicia Grijalva Covarrubias,F


ERROR 8 fechas nacimientos vacias

In [414]:
# Error 8 — verificar y documentar que hay nulos que seria benefico investigarlos (no se modifican)
print(f"Pacientes sin fecha de nacimiento: {pacientes['fecha_nacimiento'].isnull().sum()}")
pacientes[pacientes['fecha_nacimiento'].isnull()][['id_paciente', 'nombre_completo']]

Pacientes sin fecha de nacimiento: 25


,id_paciente,nombre_completo
0,1,Wilfrido Pacheco Urbina
1,2,Alejandro Rentería Bueno
16,17,Elias Zamudio Sauceda
17,18,Celia Barrios Leal
20,21,Ricardo Saiz
36,37,Ángela Gollum
37,38,Susana Loera Gaytán
43,44,Lilia Valles Dueñas
60,61,Verónica Arenas Mares
74,75,Silvano Valentín


ERROR 9 CORREOS EN PACIENTES

In [415]:
# Error 9 — reconstruir correos de pacientes
import unicodedata, re

def quitar_acentos(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def reconstruir_correo_paciente(nombre):
    partes = nombre.split()

    # ignorar partículas comunes
    particulas = {'de', 'del', 'la', 'los', 'las', 'el'}
    partes_filtradas = [p for p in partes if p.lower() not in particulas]

    primer_nombre = quitar_acentos(partes_filtradas[0].lower())

    if len(partes_filtradas) == 2:
        primer_apellido = quitar_acentos(partes_filtradas[1].lower())
    elif len(partes_filtradas) == 3:
        primer_apellido = quitar_acentos(partes_filtradas[1].lower())
    elif len(partes_filtradas) >= 4:
        primer_apellido = quitar_acentos(partes_filtradas[2].lower())
    else:
        primer_apellido = quitar_acentos(partes_filtradas[0].lower())

    return f"{primer_nombre}{primer_apellido}@hospitalvital.mx"

pacientes['correo_electronico'] = pacientes['nombre_completo'].apply(reconstruir_correo_paciente)

In [416]:
pacientes.head(20)

,id_paciente,nombre_completo,fecha_nacimiento,genero,correo_electronico,id_nacionalidad
0,1,Wilfrido Pacheco Urbina,None,F,wilfridopacheco@hospitalvital.mx,10
1,2,Alejandro Rentería Bueno,None,M,alejandrorenteria@hospitalvital.mx,10
2,3,Ignacio Hernádez Mejía,1945-01-07,M,ignaciohernadez@hospitalvital.mx,10
3,4,Karina Serrano Abreu,2018-10-01,F,karinaserrano@hospitalvital.mx,10
4,5,Patricia Ríos,1985-06-26,F,patriciarios@hospitalvital.mx,6
5,6,Ana Escobedo Sisneros,1978-06-25,F,anaescobedo@hospitalvital.mx,10
6,7,Silvia Samaniego Quintanilla,1936-04-07,F,silviasamaniego@hospitalvital.mx,10
7,8,Emilia Jaramillo Ulloa,1947-01-27,F,emiliajaramillo@hospitalvital.mx,10
8,9,Graciela Quezada Rubio,1945-10-08,F,gracielaquezada@hospitalvital.mx,10
9,10,Alicia Grijalva Covarrubias,1971-07-15,F,aliciagrijalva@hospitalvital.mx,10


ESPECIALIDADES

In [417]:
espeialidades=pd.read_sql("SELECT * FROM especialidades", engine)
espeialidades

,id_especialidad,nombre_especialidad
0,1,Cardiología
1,2,Pediatría
2,3,Dermatología
3,4,Ginecología
4,5,Neurología
5,6,Oncología


In [418]:
espeialidades["nombre_especialidad"].duplicated().any() # no hay nombres duplicados. TOOD OK

np.False_

MEDICAMENTOS

In [419]:
medicamentos=pd.read_sql("SELECT * FROM medicamentos", engine)
medicamentos

,id_medicamento,nombre_medicamento,descripcion
0,1,Paracetamol,Alivia la diarrea.
1,2,Ibuprofeno,Antiinflamatorio no esteroideo utilizado para ...
2,3,Omeprazol,Reduce la producción de ácido en el estómago.
3,4,Amoxicilina,Antibiótico para tratar infecciones bacterianas.
4,5,Loratadina,Alivia síntomas de alergia como estornudos o p...
5,6,Metformina,Ayuda a controlar los niveles de azúcar en san...
6,7,Losartán,Medicamento para controlar la presión arterial.
7,8,Ácido fólico,Suplemento usado para prevenir defectos congén...
8,9,Vitamina D,Contribuye a la absorción de calcio y fortalec...
9,10,Ranitidina,Disminuye la acidez estomacal.


ERROR 10 DESCRIPCIONES INCORRECTAS EN MEDICAMENTOS

In [420]:
medicamentos.loc[medicamentos['id_medicamento'] == 1, 'descripcion'] = 'Analgésico y antipirético para aliviar el dolor y reducir la fiebre.'
medicamentos.loc[medicamentos['id_medicamento'] == 12, 'descripcion'] = 'Broncodilatador que facilita la respiración en casos de asma o broncoespasmo.'

medicamentos.loc[medicamentos['id_medicamento'].isin([1, 12]), ['nombre_medicamento', 'descripcion']]

,nombre_medicamento,descripcion
0,Paracetamol,Analgésico y antipirético para aliviar el dolo...
11,Salbutamol,Broncodilatador que facilita la respiración en...


In [421]:
medicamentos

,id_medicamento,nombre_medicamento,descripcion
0,1,Paracetamol,Analgésico y antipirético para aliviar el dolo...
1,2,Ibuprofeno,Antiinflamatorio no esteroideo utilizado para ...
2,3,Omeprazol,Reduce la producción de ácido en el estómago.
3,4,Amoxicilina,Antibiótico para tratar infecciones bacterianas.
4,5,Loratadina,Alivia síntomas de alergia como estornudos o p...
5,6,Metformina,Ayuda a controlar los niveles de azúcar en san...
6,7,Losartán,Medicamento para controlar la presión arterial.
7,8,Ácido fólico,Suplemento usado para prevenir defectos congén...
8,9,Vitamina D,Contribuye a la absorción de calcio y fortalec...
9,10,Ranitidina,Disminuye la acidez estomacal.


RECETAS

In [422]:
recetas=pd.read_sql("SELECT * FROM recetas", engine)
recetas

,id_receta,id_paciente,id_medico,fecha_receta
0,1,5,65,2024-04-28
1,2,179,87,2023-09-20
2,3,111,23,2023-07-02
3,4,28,13,2023-09-02
4,5,135,20,2024-12-04
5,6,62,25,2023-07-15
6,7,159,68,2023-09-29
7,8,65,46,2024-12-02
8,9,69,51,2024-04-16
9,10,21,48,2023-04-19


In [423]:
recetas.isnull().any() # no hay ningun dato null

id_receta       False
id_paciente     False
id_medico       False
fecha_receta    False
dtype: bool

In [424]:
# verificar que todos los id_paciente en recetas existen en pacientes
ids_pacientes = set(pacientes['id_paciente'])
ids_medicos = set(medicos['id_medico'])

recetas_pac_invalidos = recetas[~recetas['id_paciente'].isin(ids_pacientes)]
recetas_med_invalidos = recetas[~recetas['id_medico'].isin(ids_medicos)]

print(f"Recetas con id_paciente inexistente: {len(recetas_pac_invalidos)}")
print(f"Recetas con id_medico inexistente: {len(recetas_med_invalidos)}")

Recetas con id_paciente inexistente: 0
Recetas con id_medico inexistente: 0


RECETAS MEDICAMENTOS

In [425]:
detalles_recetas=pd.read_sql("SELECT * FROM detalles_recetas", engine)
detalles_recetas

,id_detalle,id_receta,id_medicamento,dosis,frecuencia
0,1,33,8,10 ml,Una vez al día
1,2,79,7,10 ml,Cada 8 horas
2,3,18,10,1 tableta,Cada 12 horas
3,4,43,20,10 ml,Cada 12 horas
4,5,57,11,10 ml,Una vez al día
5,6,77,5,500 mg,Cada 12 horas
6,7,78,7,10 ml,Cada 12 horas
7,8,23,13,500 mg,Cada 12 horas
8,9,95,16,2 tabletas,Cada 12 horas
9,10,49,9,10 ml,Cada 12 horas


In [426]:
detalles_recetas.isnull().any() # no hay nulos

id_detalle        False
id_receta         False
id_medicamento    False
dosis             False
frecuencia        False
dtype: bool

In [427]:
ids_recetas = set(recetas['id_receta'])
ids_medicamentos = set(medicamentos['id_medicamento'])

det_receta_invalidos = detalles_recetas[~detalles_recetas['id_receta'].isin(ids_recetas)]
det_med_invalidos = detalles_recetas[~detalles_recetas['id_medicamento'].isin(ids_medicamentos)]

print(f"Detalles con id_receta inexistente: {len(det_receta_invalidos)}")
print(f"Detalles con id_medicamento inexistente: {len(det_med_invalidos)}")

Detalles con id_receta inexistente: 0
Detalles con id_medicamento inexistente: 0


TRATAMIENTOS

In [428]:
tratamientos = pd.read_sql("SELECT * FROM tratamientos", engine)

In [429]:
tratamientos

,id_tratamiento,id_paciente,descripcion_tratamiento,fecha_inicio,fecha_fin
0,1,29,Tratamiento para presión alta,2022-07-24,2024-11-22
1,2,170,,2023-10-15,2025-01-29
2,3,158,Tratamiento para presión alta,2023-05-13,2024-04-05
3,4,62,Rehabilitación postoperatoria,2022-06-24,2023-09-14
4,5,18,Tratamiento para presión alta,2022-09-30,2024-01-10
5,6,153,Tratamiento para presión alta,2022-07-02,2022-09-14
6,7,115,Terapia respiratoria,2022-10-01,2024-07-14
7,8,173,Control de alergias,2023-08-08,2024-12-03
8,9,13,Rehabilitación postoperatoria,2023-03-20,2023-05-07
9,10,190,Tratamiento para presión alta,2022-06-24,2024-08-08


In [430]:
vacias = (tratamientos['descripcion_tratamiento'] == '').sum()
print(f"Registros con descripción vacía: {vacias}")

Registros con descripción vacía: 21


In [431]:
tratamientos['descripcion_tratamiento'] = tratamientos['descripcion_tratamiento'].replace('', None)
vacias = (tratamientos['descripcion_tratamiento'] == '').sum()
print(f"Registros con descripción vacía: {vacias}")

Registros con descripción vacía: 0


In [432]:
tratamientos

,id_tratamiento,id_paciente,descripcion_tratamiento,fecha_inicio,fecha_fin
0,1,29,Tratamiento para presión alta,2022-07-24,2024-11-22
1,2,170,NaN,2023-10-15,2025-01-29
2,3,158,Tratamiento para presión alta,2023-05-13,2024-04-05
3,4,62,Rehabilitación postoperatoria,2022-06-24,2023-09-14
4,5,18,Tratamiento para presión alta,2022-09-30,2024-01-10
5,6,153,Tratamiento para presión alta,2022-07-02,2022-09-14
6,7,115,Terapia respiratoria,2022-10-01,2024-07-14
7,8,173,Control de alergias,2023-08-08,2024-12-03
8,9,13,Rehabilitación postoperatoria,2023-03-20,2023-05-07
9,10,190,Tratamiento para presión alta,2022-06-24,2024-08-08


In [433]:
# Verificar que todos los id_paciente en tratamientos existen en pacientes
trat_pac_invalidos = tratamientos[~tratamientos['id_paciente'].isin(ids_pacientes)]
print(f"Tratamientos con id_paciente inexistente: {len(trat_pac_invalidos)}")

Tratamientos con id_paciente inexistente: 0


citas

In [434]:
citas = pd.read_sql("SELECT * FROM citas", engine)
print(citas.shape)
print(citas.isnull().sum())
citas.head()

(150, 5)
id_cita            0
id_paciente        0
id_medico          0
fecha_cita         0
motivo_consulta    0
dtype: int64


,id_cita,id_paciente,id_medico,fecha_cita,motivo_consulta
0,1,183,43,2023-05-28,Consulta
1,2,122,67,2023-11-14,Control de presión
2,3,39,78,2025-01-13,Consulta pediátrica
3,4,36,42,2024-12-01,Consulta pediátrica
4,5,82,21,2023-06-16,Control de presión


In [435]:
vacias = (citas['motivo_consulta'] == '').sum()
print(f"motivo_consulta vacíos: {vacias}")

motivo_consulta vacíos: 26


In [436]:
ids_pacientes = set(pacientes['id_paciente'])
ids_medicos   = set(medicos['id_medico'])

citas_pac_inv = citas[~citas['id_paciente'].isin(ids_pacientes)]
citas_med_inv = citas[~citas['id_medico'].isin(ids_medicos)]

print(f"Citas con id_paciente inexistente: {len(citas_pac_inv)}")
print(f"Citas con id_medico inexistente:   {len(citas_med_inv)}")

Citas con id_paciente inexistente: 0
Citas con id_medico inexistente:   0


In [437]:
# Citas duplicadas (mismo paciente, médico y fecha)
duplicadas = citas[
    citas.duplicated(subset=['id_paciente', 'id_medico', 'fecha_cita'], keep=False)
].sort_values(['id_paciente', 'id_medico', 'fecha_cita'])

print(f"Citas duplicadas: {len(duplicadas)}")
print(duplicadas)

Citas duplicadas: 0
Empty DataFrame
Columns: [id_cita, id_paciente, id_medico, fecha_cita, motivo_consulta]
Index: []


In [438]:
# Fechas nulas o futuras
from datetime import date
hoy = pd.Timestamp(date.today())

citas['fecha_cita'] = pd.to_datetime(citas['fecha_cita'])
print(f"fecha_cita nulas: {citas['fecha_cita'].isna().sum()}")

futuras = citas[citas['fecha_cita'] > hoy]
print(f"Citas con fecha futura: {len(futuras)}")
print(futuras[['id_cita', 'id_paciente', 'id_medico', 'fecha_cita']])

fecha_cita nulas: 0
Citas con fecha futura: 0
Empty DataFrame
Columns: [id_cita, id_paciente, id_medico, fecha_cita]
Index: []


In [439]:
citas["motivo_consulta"].unique()

<StringArray>
[           'Consulta',  'Control de presión', 'Consulta pediátrica',
     'Revisión médica',                    '',     'Chequeo general',
     'Dolor de cabeza']
Length: 7, dtype: str

error 12 motivo consuklta cadena vacia

In [440]:
citas["motivo_consulta"].value_counts()

motivo_consulta
Consulta               26
                       26
Dolor de cabeza        25
Control de presión     21
Consulta pediátrica    21
Chequeo general        16
Revisión médica        15
Name: count, dtype: int64

In [441]:
citas['motivo_consulta'] = citas['motivo_consulta'].replace('', None)

In [442]:
citas["motivo_consulta"].value_counts()

motivo_consulta
Consulta               26
Dolor de cabeza        25
Control de presión     21
Consulta pediátrica    21
Chequeo general        16
Revisión médica        15
Name: count, dtype: int64

error 13 valres inconsistentes en motivo consulta

In [443]:
mapeo_motivo = {
    'Consulta':           'Consulta general',
    'Revisión médica':    'Consulta general',
    'Chequeo general':    'Consulta general',
}
citas['motivo_consulta'] = citas['motivo_consulta'].replace(mapeo_motivo)

In [444]:
citas['motivo_consulta'].value_counts()

motivo_consulta
Consulta general       57
Dolor de cabeza        25
Control de presión     21
Consulta pediátrica    21
Name: count, dtype: int64

In [445]:
citas

,id_cita,id_paciente,id_medico,fecha_cita,motivo_consulta
0,1,183,43,2023-05-28,Consulta general
1,2,122,67,2023-11-14,Control de presión
2,3,39,78,2025-01-13,Consulta pediátrica
3,4,36,42,2024-12-01,Consulta pediátrica
4,5,82,21,2023-06-16,Control de presión
5,6,158,95,2024-04-27,Consulta general
6,7,77,76,2024-11-17,Consulta general
7,8,130,66,2024-05-22,Consulta pediátrica
8,9,126,91,2025-01-06,Consulta pediátrica
9,10,77,61,2024-10-23,Consulta general


EXPORTACION DE LOS DATAFRAMES A LA BD

In [447]:
from sqlalchemy import text

nacionalidades_df = nacionalidades.rename(columns={
    'id_nacionalidad': 'id',
    'nombre_nacionalidad': 'nombre'
})

medicos['id_nacionalidad'] = medicos['id_nacionalidad'].astype(int)
pacientes['id_nacionalidad'] = pacientes['id_nacionalidad'].astype(int)

with engine.connect() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS=0;"))

    conn.execute(text("TRUNCATE TABLE detalles_recetas;"))
    conn.execute(text("TRUNCATE TABLE recetas;"))
    conn.execute(text("TRUNCATE TABLE citas;"))
    conn.execute(text("TRUNCATE TABLE tratamientos;"))
    conn.execute(text("TRUNCATE TABLE pacientes;"))
    conn.execute(text("TRUNCATE TABLE medicos;"))
    conn.execute(text("TRUNCATE TABLE medicamentos;"))

    conn.execute(text("DROP TABLE IF EXISTS nacionalidades;"))

    conn.execute(text("SET FOREIGN_KEY_CHECKS=1;"))
    conn.commit()

with engine.connect() as conn:
    try:
        conn.execute(text("ALTER TABLE medicos ADD COLUMN id_nacionalidad INT;"))
    except:
        pass

    try:
        conn.execute(text("ALTER TABLE pacientes ADD COLUMN id_nacionalidad INT;"))
    except:
        pass

    try:
        conn.execute(text("ALTER TABLE medicos DROP COLUMN nacionalidad;"))
    except:
        pass

    try:
        conn.execute(text("ALTER TABLE pacientes DROP COLUMN nacionalidad;"))
    except:
        pass

    conn.execute(text("""
        CREATE TABLE nacionalidades (
            id INT PRIMARY KEY,
            nombre VARCHAR(100) NOT NULL UNIQUE
        );
    """))

    conn.commit()

nacionalidades_df.to_sql('nacionalidades', con=engine, if_exists='append', index=False)

medicamentos.to_sql('medicamentos', con=engine, if_exists='append', index=False)

medicos.to_sql('medicos', con=engine, if_exists='append', index=False)
pacientes.to_sql('pacientes', con=engine, if_exists='append', index=False)

tratamientos.to_sql('tratamientos', con=engine, if_exists='append', index=False)
citas.to_sql('citas', con=engine, if_exists='append', index=False)
recetas.to_sql('recetas', con=engine, if_exists='append', index=False)

detalles_recetas.to_sql('detalles_recetas', con=engine, if_exists='append', index=False)

with engine.connect() as conn:
    try:
        conn.execute(text("ALTER TABLE medicos DROP FOREIGN KEY fk_medicos_nacionalidad;"))
    except:
        pass

    try:
        conn.execute(text("ALTER TABLE pacientes DROP FOREIGN KEY fk_pacientes_nacionalidad;"))
    except:
        pass

    conn.execute(text("""
        ALTER TABLE medicos
        ADD CONSTRAINT fk_medicos_nacionalidad
        FOREIGN KEY (id_nacionalidad) REFERENCES nacionalidades(id);
    """))

    conn.execute(text("""
        ALTER TABLE pacientes
        ADD CONSTRAINT fk_pacientes_nacionalidad
        FOREIGN KEY (id_nacionalidad) REFERENCES nacionalidades(id);
    """))

    conn.commit()